# CASMI26 — Classical ML molecule retrieval and reranking
Author: project owner (fill in your name before submission).

Input: LC-MS/MS spectra. Output: one ranked list of at most 25 unique SMILES per molecule.
Metric: molecule-level MRR@25. Remove stereochemistry BEFORE RDKit tautomer canonicalization and InChIKey14, matching the [official metric](https://www.kaggle.com/code/metric/casmi-mean-reciprocal-rank).
Architecture: sparse coarse retrieval + exact peak matching + neutral-mass candidate pool + Random Forest / XGBoost reranking.

References: [competition](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra),
[RDKit](https://www.rdkit.org/docs/), [Random Forest (Breiman, 2001)](https://doi.org/10.1023/A:1010933404324),
[XGBoost (Chen and Guestrin, 2016)](https://doi.org/10.1145/2939672.2939785).
Implementation is original. Retrieval cannot identify an exact structure absent from the candidate pool.

## Full-query training with GPU XGBoost
Use the **competition**, the existing **reference/pool assets**, `chemistry.sqlite`,
and the **full CUDA xgboost 3.2.0 wheel**. Internet OFF. The old 1,000-query model
is only an input container for reusable references; its weights are never used.

All valid canonical identities become queries (up to one hash-selected distinct
spectrum per ion mode). This does not mean 2.5 million separate spectrum queries.
Identities without independent A references remain in training as B queries.
Reference spectra and expanded COCONUT candidates remain unchanged.

CPU feature generation can exceed one session. The notebook saves atomic chunks
and a checkpoint; attach the previous notebook output to continue. No claim that
full preparation or GPU training has completed is made before the logs prove it.


In [ ]:
# Optional settings: edit before Run All.
import os
STAGE = os.environ.get('CASMI_FULL_STAGE', 'train')  # prepare | train | all
PREPARE_HOURS = float(os.environ.get('CASMI_PREPARE_HOURS', '7'))
VALIDATION_SIZE = int(os.environ.get('CASMI_VALIDATION_SIZE', '1000'))
BATCH_QUERIES = int(os.environ.get('CASMI_BATCH_QUERIES', '128'))
# Resume: attach previous notebook output; set this to its full-training folder.
RESUME_FROM = os.environ.get('CASMI_RESUME_FROM')
SYNTHETIC = os.environ.get('CASMI_ALLOW_SYNTHETIC', '0') == '1'
DEVICE = os.environ.get('CASMI_TRAIN_DEVICE', 'cuda')
if STAGE not in {'prepare', 'train', 'all'}:
    raise ValueError('Unknown stage')
if DEVICE != 'cuda' and not SYNTHETIC and STAGE != 'prepare':
    raise ValueError('Production fitting requires CUDA; CPU is allowed for preparation only')
from pathlib import Path
import tempfile, zipfile
if not SYNTHETIC and 'CASMI_TRAIN_WHEELS' not in os.environ:
    archives = sorted(Path(os.environ.get('CASMI_INPUT_ROOT', '/kaggle/input')).rglob('casmi26_training_inputs.zip'))
    if len(archives) == 1:
        training_inputs = Path(tempfile.mkdtemp(prefix='casmi26_training_'))
        with zipfile.ZipFile(archives[0]) as archive:
            for member in archive.infolist():
                target = (training_inputs/member.filename).resolve()
                if target != training_inputs.resolve() and training_inputs.resolve() not in target.parents:
                    raise ValueError('Unsafe training archive path')
            archive.extractall(training_inputs)
        os.environ['CASMI_TRAIN_WHEELS'] = str(training_inputs/'wheels')
        os.environ.setdefault('CASMI_STRUCTURE_CACHE', str(training_inputs/'chemistry.sqlite'))


In [ ]:
# Synthetic tests use existing local CPU dependencies.
if SYNTHETIC:
    # Source bundled at build time; no network or local repository needed.
    import os, sys, io, base64, zipfile, tempfile, random, json, shutil
    from pathlib import Path
    os.environ.setdefault('OMP_NUM_THREADS', '4')
    os.environ.setdefault('OPENBLAS_NUM_THREADS', '4')
    _PACKAGE_B64 = '...'
    _package_dir = Path(tempfile.mkdtemp(prefix='casmi26_source_'))
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(_PACKAGE_B64))) as _zip:
        _zip.extractall(_package_dir)
    sys.path.insert(0, str(_package_dir))
    # Accept either an expanded Dataset or the single-file upload package.
    if 'CASMI_ASSETS' not in os.environ:
        _input_root = Path(os.environ.get('CASMI_INPUT_ROOT','/kaggle/input'))
        _existing = [p for p in _input_root.rglob('manifest.json') if (p.parent/'pool.parquet').is_file()]
        if not _existing:
            _archives = sorted(_input_root.rglob('casmi26_assets.zip'))
            if len(_archives) == 1:
                _unpacked = Path(tempfile.mkdtemp(prefix='casmi26_assets_'))
                with zipfile.ZipFile(_archives[0]) as _asset_zip:
                    for _member in _asset_zip.infolist():
                        _target = (_unpacked/_member.filename).resolve()
                        if _target != _unpacked.resolve() and _unpacked.resolve() not in _target.parents:
                            raise ValueError('Unsafe path in asset archive')
                    _asset_zip.extractall(_unpacked)
                os.environ['CASMI_ASSETS'] = str(_unpacked)
                print('Extracted attached asset package to scratch storage')
    
    # Install only compatible attached wheels; pip never contacts an index.
    import importlib.metadata as _metadata
    import subprocess as _subprocess
    from packaging.tags import sys_tags as _sys_tags
    from packaging.utils import parse_wheel_filename as _parse_wheel, canonicalize_name as _package_name
    from packaging.version import Version as _Version
    os.environ['PIP_DISABLE_PIP_VERSION_CHECK'] = '1'
    _tags = set(_sys_tags())
    for _dist, _target, _module in [('rdkit','2026.3.3','rdkit'), ('scikit-learn','1.7.2','sklearn'), ('xgboost','3.2.0','xgboost')]:
        try:
            _actual = _metadata.version('xgboost-cpu') if _dist == 'xgboost' and any(d.metadata['Name'] == 'xgboost-cpu' for d in _metadata.distributions()) else _metadata.version(_dist)
        except _metadata.PackageNotFoundError:
            _actual = None
        if _actual == _target or (_dist == 'rdkit' and _actual and _Version(_actual).release[:2] == _Version(_target).release[:2]):
            continue
        if _module in sys.modules:
            raise RuntimeError('Restart the kernel before installing the required offline version of '+_dist)
        _roots = [Path(os.environ.get('CASMI_INPUT_ROOT','/kaggle/input'))]
        if 'CASMI_ASSETS' in os.environ:
            _roots.append(Path(os.environ['CASMI_ASSETS']))
        if 'CASMI_TRAIN_WHEELS' in os.environ:
            _roots.append(Path(os.environ['CASMI_TRAIN_WHEELS']))
        _wheels = []
        for _root in _roots:
            for _wheel in _root.rglob('*.whl'):
                try:
                    _name, _version, _, _wheel_tags = _parse_wheel(_wheel.name)
                except ValueError:
                    continue
                if (_name == _package_name(_dist) or (_dist == 'xgboost' and _name == 'xgboost-cpu')) and _version == _Version(_target) and _tags.intersection(_wheel_tags):
                    _wheels.append(_wheel)
        if not _wheels:
            raise RuntimeError('Attach export/wheels with compatible offline '+_dist+'=='+_target+' for this Python/platform; current='+str(_actual))
        _wheels.sort(key=lambda p: ('xgboost_cpu' not in p.name, str(p)))
        if _dist == 'xgboost' and _wheels[0].name.startswith('xgboost_cpu'):
            try:
                _metadata.version('xgboost')
            except _metadata.PackageNotFoundError:
                pass
            else:
                _subprocess.run([sys.executable,'-m','pip','uninstall','-y','xgboost'],check=True,capture_output=True,text=True)
        _subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps',str(_wheels[0])],check=True,capture_output=True,text=True)
        print('Installed offline dependency:', _dist, _target)
    
    import numpy as np, psutil, rdkit
    from casmi26.assets import versions
    random.seed(2026); np.random.seed(2026)
    print('Package versions:', versions())
    print('Runtime:', {'cpu_count': os.cpu_count(), 'ram_total': psutil.virtual_memory().total,
                       'ram_available': psutil.virtual_memory().available})
else:
    # Source bundled at build time; no network or local repository needed.
    import os, sys, io, base64, zipfile, tempfile, random, json, shutil
    from pathlib import Path
    os.environ.setdefault('OMP_NUM_THREADS', '4')
    os.environ.setdefault('OPENBLAS_NUM_THREADS', '4')
    _PACKAGE_B64 = '...'
    _package_dir = Path(tempfile.mkdtemp(prefix='casmi26_source_'))
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(_PACKAGE_B64))) as _zip:
        _zip.extractall(_package_dir)
    sys.path.insert(0, str(_package_dir))
    # Accept either an expanded Dataset or the single-file upload package.
    if 'CASMI_ASSETS' not in os.environ:
        _input_root = Path(os.environ.get('CASMI_INPUT_ROOT','/kaggle/input'))
        _existing = [p for p in _input_root.rglob('manifest.json') if (p.parent/'pool.parquet').is_file()]
        if not _existing:
            _archives = sorted(_input_root.rglob('casmi26_assets.zip'))
            if len(_archives) == 1:
                _unpacked = Path(tempfile.mkdtemp(prefix='casmi26_assets_'))
                with zipfile.ZipFile(_archives[0]) as _asset_zip:
                    for _member in _asset_zip.infolist():
                        _target = (_unpacked/_member.filename).resolve()
                        if _target != _unpacked.resolve() and _unpacked.resolve() not in _target.parents:
                            raise ValueError('Unsafe path in asset archive')
                    _asset_zip.extractall(_unpacked)
                os.environ['CASMI_ASSETS'] = str(_unpacked)
                print('Extracted attached asset package to scratch storage')
    
    # Install only compatible attached wheels; pip never contacts an index.
    import importlib.metadata as _metadata
    import subprocess as _subprocess
    from packaging.tags import sys_tags as _sys_tags
    from packaging.utils import parse_wheel_filename as _parse_wheel, canonicalize_name as _package_name
    from packaging.version import Version as _Version
    os.environ['PIP_DISABLE_PIP_VERSION_CHECK'] = '1'
    _tags = set(_sys_tags())
    for _dist, _target, _module in [('nvidia-nccl-cu12','2.29.3','nvidia.nccl'), ('rdkit','2026.3.3','rdkit'), ('scikit-learn','1.7.2','sklearn'), ('xgboost','3.2.0','xgboost')]:
        try:
            _actual = _metadata.version(_dist)
        except _metadata.PackageNotFoundError:
            _actual = None
        if _dist == 'nvidia-nccl-cu12' and _actual:
            print('Keeping installed NCCL:', _actual)
            continue
        if (_actual == _target and not (_dist == 'xgboost' and any(d.metadata['Name'] == 'xgboost-cpu' for d in _metadata.distributions()))) or (_dist == 'rdkit' and _actual and _Version(_actual).release[:2] == _Version(_target).release[:2]):
            continue
        if _module in sys.modules:
            raise RuntimeError('Restart the kernel before installing the required offline version of '+_dist)
        _roots = [Path(os.environ.get('CASMI_INPUT_ROOT','/kaggle/input'))]
        if 'CASMI_ASSETS' in os.environ:
            _roots.append(Path(os.environ['CASMI_ASSETS']))
        if 'CASMI_TRAIN_WHEELS' in os.environ:
            _roots.append(Path(os.environ['CASMI_TRAIN_WHEELS']))
        _wheels = []
        for _root in _roots:
            for _wheel in _root.rglob('*.whl'):
                try:
                    _name, _version, _, _wheel_tags = _parse_wheel(_wheel.name)
                except ValueError:
                    continue
                if (_name == _package_name(_dist)) and _version == _Version(_target) and _tags.intersection(_wheel_tags):
                    _wheels.append(_wheel)
        if not _wheels:
            raise RuntimeError('Attach export/wheels with compatible offline '+_dist+'=='+_target+' for this Python/platform; current='+str(_actual))
        _wheels.sort(key=lambda p: ('xgboost_cpu' not in p.name, str(p)))
        if _dist == 'xgboost' and _wheels[0].name.startswith('xgboost_cpu'):
            try:
                _metadata.version('xgboost')
            except _metadata.PackageNotFoundError:
                pass
            else:
                _subprocess.run([sys.executable,'-m','pip','uninstall','-y','xgboost'],check=True,capture_output=True,text=True)
        if _dist == 'xgboost':
            _subprocess.run([sys.executable,'-m','pip','uninstall','-y','xgboost-cpu'],check=True,capture_output=True,text=True)
        _subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps',str(_wheels[0])],check=True,capture_output=True,text=True)
        print('Installed offline dependency:', _dist, _target)
    
    import numpy as np, psutil, rdkit
    from casmi26.assets import versions
    random.seed(2026); np.random.seed(2026)
    print('Package versions:', versions())
    print('Runtime:', {'cpu_count': os.cpu_count(), 'ram_total': psutil.virtual_memory().total,
                       'ram_available': psutil.virtual_memory().available})

In [ ]:
from casmi26.assets import locate_assets
from casmi26.chemistry import configure_structure_cache
from casmi26.schema import Config
from casmi26.full_training import prepare_features, train_and_export, require_gpu
INPUT_ROOT = Path(os.environ.get('CASMI_INPUT_ROOT', '/kaggle/input'))
ASSETS = Path(os.environ['CASMI_ASSETS']) if 'CASMI_ASSETS' in os.environ else locate_assets(INPUT_ROOT)
if 'CASMI_TRAIN_PATH' in os.environ:
    TRAIN_PATH = Path(os.environ['CASMI_TRAIN_PATH'])
else:
    matches = sorted(INPUT_ROOT.rglob('train.parquet'))
    if len(matches) != 1:
        raise FileNotFoundError('Attach exactly one competition train.parquet or set CASMI_TRAIN_PATH')
    TRAIN_PATH = matches[0]
WORK = Path(os.environ.get('CASMI_TRAIN_WORK', '/kaggle/working/full-training')).resolve()
if not RESUME_FROM and not SYNTHETIC:
    _auto_candidates = [p.parent for p in INPUT_ROOT.rglob('checkpoint.json') if (p.parent/'features').is_dir()]
    if len(_auto_candidates) == 1:
        RESUME_FROM = str(_auto_candidates[0])
        print('Auto-detected resume folder from attached input:', RESUME_FROM)
if RESUME_FROM:
    source = Path(RESUME_FROM).resolve()
    if not (source/'checkpoint.json').is_file():
        _found = list(source.rglob('checkpoint.json'))
        if _found:
            source = _found[0].parent
        else:
            raise FileNotFoundError(f'Resume folder must contain checkpoint.json (checked: {source})')
    print('Resuming from:', source)
    if WORK != source and not WORK.exists():
        size = sum(p.stat().st_size for p in source.rglob('*') if p.is_file())
        if size > 12*2**30 or size > shutil.disk_usage(WORK.parent).free*.7:
            raise MemoryError('Resume copy exceeds working budget')
        shutil.copytree(source, WORK)
cfg = Config(batch_rows=100 if SYNTHETIC else 5000)
cache = os.environ.get('CASMI_STRUCTURE_CACHE')
if not cache:
    candidates = sorted(INPUT_ROOT.rglob('chemistry.sqlite'))
    cache = str(candidates[0]) if len(candidates) == 1 else None
if cache:
    try:
        configure_structure_cache(cache)
    except Exception:
        import shutil, tempfile
        _tmp_cache = Path(tempfile.gettempdir()) / Path(cache).name
        if not _tmp_cache.is_file() or _tmp_cache.stat().st_size != Path(cache).stat().st_size:
            shutil.copy2(cache, _tmp_cache)
        configure_structure_cache(str(_tmp_cache))
elif not SYNTHETIC:
    raise FileNotFoundError('Attach metric-v14 chemistry.sqlite for full-scale query selection')
if STAGE in {'train', 'all'} and not SYNTHETIC:
    print('Verified GPU:', require_gpu())
print({'stage': STAGE, 'train': str(TRAIN_PATH), 'assets': str(ASSETS), 'work': str(WORK),
       'device': DEVICE, 'all_queries': True, 'validation_size': VALIDATION_SIZE})


## 1. Prepare or resume features (CPU)
Query identities are split by a seeded hash. Query content hashes and duplicates
are excluded globally. Training masks validation identities; B/C additionally
mask every reference sharing the query canonical key. C removes the pool key.
Query labels are used only to create targets, never as inference features.

Checkpoint identity covers data, assets, source code, config and split parameters.
Completed chunks are checksum-verified before reuse. A time limit is checked
between batches; allow headroom for a final batch and training. If preparation
pauses, save the output and resume; no incomplete final model is exported.


In [ ]:
if STAGE in {'prepare', 'all'}:
    state = prepare_features(TRAIN_PATH, ASSETS, WORK, cfg, VALIDATION_SIZE,
                             BATCH_QUERIES, PREPARE_HOURS,
                             kind='synthetic' if SYNTHETIC else 'real')
else:
    state = json.loads((WORK/'checkpoint.json').read_text())
print({'complete': state['complete'], 'finished_chunks': len(state['chunks']),
       'expected_chunks': state.get('expected_chunks'), 'query_stats': state.get('query_stats'),
       'resources': state.get('resources_last_session')})


## 2. Fit GPU XGBoost, evaluate, then refit all queries
The iterator streams feature chunks into a QuantileDMatrix; the full dense table
is never concatenated. The quantized matrix still needs to fit host/GPU memory.
Training verifies the actual CUDA device and rejects silent CPU fallback.
Fixed hyperparameters, equal total pair weight per query, positive-class weighting.
No early stopping or hyperparameter search is claimed.

The first model sees training queries only. A/B/C MRR@25 is measured and saved
before refitting. A is reported only where an independent reference remains;
B/C cover every validation identity. The final model is refitted on every valid
query using A/B features; validation metrics belong to the pre-refit checkpoint.
The final model is saved for CPU inference. Preparation reference masks remain
conservative after refit. The full-data RF/mask/pool ablation is not rerun here;
the earlier 12-experiment baseline is documented separately.


In [ ]:
report = None
if STAGE in {'train', 'all'} and state['complete']:
    report = train_and_export(ASSETS, WORK, device=DEVICE, refit_all=True, small=SYNTHETIC)
    print(json.dumps(report, indent=2))
elif not state['complete']:
    print('PREPARATION PAUSED. Save output and resume from this full-training folder. No final model exported.')
else:
    print('Features complete. Run stage=train with a GPU to fit and export.')


In [ ]:
if report is not None:
    from casmi26.assets import verify_assets
    final_manifest = verify_assets(WORK/'export')
    assert final_manifest['model_family'] == 'xgboost'
    print('EXPORT READY:', WORK/'export')
    print('Actual final A/B query counts:', final_manifest['training_query_distribution'])
    print('Held-out metrics:', report['validation'])
    print('Attach ONLY this export as assets for CASMI26_Submission.ipynb; CPU; Internet OFF.')
else:
    print('Checkpoint output:', WORK)


## Handoff
Keep the whole `full-training` folder for resume. For submission, create a private
Dataset from **only `full-training/export`**, including model, reference, pool,
wheels and manifest. Attach that Dataset to `CASMI26_Submission.ipynb` together
with the competition input. Remove old baseline assets to avoid ambiguous input
discovery. CPU, Internet OFF. Check the CSV and submit yourself.

The old baseline ZIP remains a fallback. Neither this notebook nor the agent
uploads or submits automatically. GPU success, full-data MRR and total runtime
remain unverified until an actual full Kaggle run finishes.

Implementation reference: [XGBoost quantile data iterator](https://xgboost.readthedocs.io/en/release_3.0.0/python/examples/quantile_data_iterator.html),
[GPU support and CPU model portability](https://xgboost.readthedocs.io/en/release_3.2.0/gpu/index.html).
